<a href="https://colab.research.google.com/github/2anoFIAP/AED-KPOP-CP-DATASCIENCE/blob/main/CP_AED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AED — shows de K-pop

## 1. Contextualização

### Pergunta central
Quais fatores estão associados à lucratividade da operação dos shows de K-pop?

### Contexto e limites
A base contém shows programados entre 2023 e 2025. Os nomes dos grupos são reais, mas os eventos e indicadores são simulados para fins educacionais.

Os resultados financeiros representam a operação de realização dos shows, não o lucro total dos grupos, artistas ou agências.

### Unidade de análise
Cada linha representa um show programado, inclusive quando cancelado. Também serão utilizadas agregações por turnê para avaliar o resultado da programação.

### Objetivo
Investigar relações entre características dos shows, contexto de mercado e desempenho financeiro, desenvolvendo pelo menos três insights e três ações vinculadas às evidências.

### Cuidados metodológicos
- Manter cancelamentos na avaliação financeira da operação.
- Não substituir automaticamente dados ausentes por zero.
- Distinguir lucro absoluto, margem e frequência de prejuízo.
- Considerar a dependência entre shows do mesmo grupo ou turnê.
- Interpretar associações sem presumir causalidade.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(
    "kpop_shows.csv",
    encoding="utf-8-sig",
    parse_dates=["data_show"]
)
df.head(5)

,id_show,dados_simulados,grupo,integrante_referencia,ano_estreia_grupo,data_show,ano,id_turne,ordem_show_turne,shows_planejados_turne,...,custo_marketing_usd,custo_produtos_usd,repasse_artistas_usd,comissao_promotor_usd,impostos_usd,rateio_abertura_turne_usd,receita_total_usd,custo_total_usd,lucro_operacional_usd,satisfacao_publico_0a10
0,K00001,True,TWICE,Jihyo,2015,2023-02-02,2023,T01-2023-1,1,8,...,54473.49,31141.86,54022.94,43156.84,32674.04,12555.79,523755.27,292376.31,231378.96,8.0
1,K00002,True,TWICE,Jihyo,2015,2023-02-05,2023,T01-2023-1,2,8,...,65081.42,91751.78,268334.08,198778.53,226494.08,12555.79,2357054.06,972198.04,1384856.02,8.5
2,K00003,True,TWICE,Jihyo,2015,2023-02-08,2023,T01-2023-1,3,8,...,86207.86,48788.28,179820.49,199192.56,68062.84,12555.79,1659454.20,672432.22,987021.98,7.2
3,K00004,True,TWICE,Jihyo,2015,2023-02-14,2023,T01-2023-1,4,8,...,50474.08,86804.44,310560.84,381878.22,137238.80,12555.79,2808116.01,1177750.74,1630365.27,8.4
4,K00005,True,TWICE,Jihyo,2015,2023-02-17,2023,T01-2023-1,5,8,...,40516.19,76202.43,226245.20,275082.71,162844.01,12555.79,1912873.37,934102.64,978770.73,7.5


## 2. Compreensão e inspeção dos dados

Antes de qualquer limpeza, vamos verificar a estrutura da base, os tipos das variáveis, a cobertura dos registros, possíveis duplicidades e os valores ausentes.

In [4]:
df_original = df.copy(deep=True)

print(f"Dimensões: {df.shape[0]:,} linhas e {df.shape[1]} colunas")
print(f"Grupos: {df['grupo'].nunique()}")
print(f"Turnês: {df['id_turne'].nunique()}")
print(f"Período: {df['data_show'].min():%d/%m/%Y}"
    f" a {df['data_show'].max():%d/%m/%Y}")

print("\nTipos e preenchimento das colunas:")
df.info()

print("\nStatus dos shows:")
display(df["status_show"].value_counts(dropna=False).rename_axis("status_show").to_frame("quantidade"))

print("\nVerificação de duplicidades:")
print(f"Linhas inteiramente duplicadas: {df.duplicated().sum()}")
print(f"IDs de show ausentes: {df['id_show'].isna().sum()}")
print(f"IDs de show duplicados: {df['id_show'].duplicated().sum()}")

ausentes = pd.DataFrame({
    "quantidade_ausente": df.isna().sum(),
    "percentual_ausente": df.isna().mean().mul(100)
})

ausentes = (
    ausentes.loc[ausentes["quantidade_ausente"] > 0]
    .sort_values("quantidade_ausente", ascending=False)
)

print("\nColunas com dados ausentes:")
display(ausentes.round(2))

Dimensões: 1,902 linhas e 48 colunas
Grupos: 20
Turnês: 180
Período: 20/01/2023 a 07/09/2025

Tipos e preenchimento das colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1902 entries, 0 to 1901
Data columns (total 48 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   id_show                          1902 non-null   object        
 1   dados_simulados                  1902 non-null   bool          
 2   grupo                            1902 non-null   object        
 3   integrante_referencia            1902 non-null   object        
 4   ano_estreia_grupo                1902 non-null   int64         
 5   data_show                        1902 non-null   datetime64[ns]
 6   ano                              1902 non-null   int64         
 7   id_turne                         1902 non-null   object        
 8   ordem_show_turne                 1902 non-null   int64         
 9   

,quantidade
status_show,
realizado,1878
cancelado,24



Verificação de duplicidades:
Linhas inteiramente duplicadas: 0
IDs de show ausentes: 0
IDs de show duplicados: 0

Colunas com dados ausentes:


,quantidade_ausente,percentual_ausente
intervalo_show_anterior_dias,180,9.46
satisfacao_publico_0a10,88,4.63
seguidores_locais_milhoes,67,3.52
streams_locais_30_dias_milhoes,58,3.05


### 2.1. Investigação dos dados ausentes

As ausências serão examinadas conforme o significado de cada variável.

- O intervalo desde o show anterior não se aplica à primeira data da turnê.
- A satisfação não se aplica aos shows cancelados; sua ausência em shows realizados representa falta de avaliação.
- Seguidores e streams possuem medições incompletas, que não devem ser interpretadas como zero.

Antes de definir tratamentos, verificaremos se os padrões observados são compatíveis com essas interpretações.